# Phytoplankton Phenology Study in the Northwest Atlantic Ocean

#### Import Python Libraries and datasets

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.patheffects as path_effects
import cartopy
import pandas as pd
from shapely.ops import unary_union
import cartopy.feature as cfeature
import geopandas as gpd
from shapely.geometry import mapping
from shapely.geometry import Polygon
import rioxarray
#import sys
#sys.path.append(r'C:\Users\grace.davis\Documents\GitHub\RESOURCES\python')
#import utilities
#from utilities import get_prod_files
import cmocean
from matplotlib.colors import LogNorm
import cartopy.crs as crs
import statsmodels as sm
from statsmodels import nonparametric
from statsmodels.nonparametric import smoothers_lowess
import scipy
from scipy import signal
from scipy.signal import find_peaks
from scipy.signal import argrelextrema
from scipy.signal import savgol_filter
from scipy.integrate import trapezoid
from collections import Counter
import plotly.express as px
import seaborn as sns
import calendar
from scipy import stats
from collections import defaultdict

In [ ]:
#Daily chlorophyll data and regional zarr files
daily_data = xr.open_zarr(r'C:\Users\grace\OneDrive\Documents\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8_COMBINED.zarr',consolidated=True)
MABS = xr.open_zarr(r'C:\Users\grace\OneDrive\Documents\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\MABS_D8.zarr')
MABN = xr.open_zarr(r'C:\Users\grace\OneDrive\Documents\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\MABN_D8.zarr')
GB = xr.open_zarr(r'C:\Users\grace\OneDrive\Documents\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\GB_D8.zarr')
GOMW = xr.open_zarr(r'C:\Users\grace\OneDrive\Documents\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\GOMW_D8.zarr')
GOME = xr.open_zarr(r'C:\Users\grace\OneDrive\Documents\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\GOME_D8.zarr')

In [ ]:
#Region shapefiles
shapefile = gpd.read_file('https://github.com/hsynan/READ-EDAB-Synan_hydrographic_climatologies/raw/refs/heads/main/data/shapefiles/NES_5REGIONS.zip')
MAB_south_loc = shapefile.iloc[[0]].set_crs("epsg:4326", inplace=True)
MAB_north_loc = shapefile.iloc[[1]].set_crs("epsg:4326", inplace=True)
GB_whole_loc = shapefile.iloc[[2]].set_crs("epsg:4326", inplace=True)
GOM_west_loc = shapefile.iloc[[3]].set_crs("epsg:4326", inplace=True)
GOM_east_loc = shapefile.iloc[[4]].set_crs("epsg:4326", inplace=True)
NES = shapefile.dissolve()

In [ ]:
#Metric csv files for plotting
summary_MABS = pd.read_csv(r'C:\Users\grace\OneDrive\Documents\GitHub\phytoplankton_Hollings_project\Datasets\MABS_Summary_Stats.csv')
summary_MABN = pd.read_csv(r'C:\Users\grace\OneDrive\Documents\GitHub\phytoplankton_Hollings_project\Datasets\MABN_Summary_Stats.csv')
summary_GB = pd.read_csv(r'C:\Users\grace\OneDrive\Documents\GitHub\phytoplankton_Hollings_project\Datasets\GB_Summary_Stats.csv')
summary_GOMW = pd.read_csv(r'C:\Users\grace\OneDrive\Documents\GitHub\phytoplankton_Hollings_project\Datasets\GOMW_Summary_Stats.csv')
summary_GOME = pd.read_csv(r'C:\Users\grace\OneDrive\Documents\GitHub\phytoplankton_Hollings_project\Datasets\GOME_Summary_Stats.csv')
bloom_MABS = pd.read_csv(r'C:\Users\grace\OneDrive\Documents\GitHub\phytoplankton_Hollings_project\Datasets\MABS_Bloom_Metrics.csv')
bloom_MABN = pd.read_csv(r'C:\Users\grace\OneDrive\Documents\GitHub\phytoplankton_Hollings_project\Datasets\MABN_Bloom_Metrics.csv')
bloom_GB = pd.read_csv(r'C:\Users\grace\OneDrive\Documents\GitHub\phytoplankton_Hollings_project\Datasets\GB_Bloom_Metrics.csv')
bloom_GOMW = pd.read_csv(r'C:\Users\grace\OneDrive\Documents\GitHub\phytoplankton_Hollings_project\Datasets\GOMW_Bloom_Metrics.csv')
bloom_GOME = pd.read_csv(r'C:\Users\grace\OneDrive\Documents\GitHub\phytoplankton_Hollings_project\Datasets\GOME_Bloom_Metrics.csv')

### Plotting the heatmaps for each region

In [ ]:
data = [summary_MABS,summary_MABN,summary_GB,summary_GOMW,summary_GOME]
region_acro = ['MABS','MABN','GB','GOMW','GOME']
region_title = ['Middle Atlantic Bight South','Middle Atlantic Bight North','Georges Bank','Gulf of Maine West','Gulf of Maine East']
for x in range(5):
    data_set = data[x]
    data_for_hm = {
        'Year': data_set['Year'],
        'January': data_set['January Integrated chl'],
        'February': data_set['February Integrated chl'],
        'March': data_set['March Integrated chl'],
        'April': data_set['April Integrated chl'],
        'May': data_set['May Integrated chl'],
        'June': data_set['June Integrated chl'],
        'July': data_set['July Integrated chl'],
        'August': data_set['August Integrated chl'],
        'September': data_set['September Integrated chl'],
        'October': data_set['October Integrated chl'],
        'November': data_set['November Integrated chl'],
        'December': data_set['December Integrated chl'],
    }
    df = pd.DataFrame(data_for_hm)
    heatmap_data = df.set_index('Year') # Makes year the x axis

    plt.figure(figsize=(18,6))
    sns.heatmap(
        heatmap_data,
        cmap='Greens',
        annot=False,
        fmt=".1f",
        linewidths=0.5,
        cbar_kws={'label': "Integrated Chlorophyll ($mg/m^3 * day$)"}
    )
    plt.title(f"Monthly Integrated Chlorophyll by Year for {region_title[x]}")
    plt.xlabel("Year")
    plt.ylabel("month")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()
    plt.savefig(rf'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\Figures\{region_acro[x]}_heatmap_chl_intensity')

### Plotting the number of events per year per region (bar chart)

#### Single line bar chart

![Bar chart of the number of blooms per year in each region](Figures\Annual_Metrics\Annual_Blooms_Bars.png)

In [ ]:
fig, ax = plt.subplots(figsize=(18,8))
width=0.14
x=summary_MABS['Year']
ax.bar(x-2*width,summary_MABS['Number_of_blooms'],color='gold',label='MAB South',edgecolor='black',width=width)
ax.bar(x-width,summary_MABN['Number_of_blooms'],color='cyan',label='MAB North',edgecolor='black',width=width)
ax.bar(x,summary_GB['Number_of_blooms'],color='darkorange',label='Georges Bank',edgecolor='black',width=width)
ax.bar(x+width,summary_GOMW['Number_of_blooms'],color='mediumorchid',label='GOM West',edgecolor='black',width=width)
ax.bar(x+2*width,summary_GOME['Number_of_blooms'],color='dodgerblue',label='GOM East',edgecolor='black',width=width)
ax.set_xticks(x)
tick_positions = [0,1,2,3,4,5,6]
tick_labels = ['0','1','2','3','4','5','6']
ax.set_yticks(tick_positions,labels=tick_labels)
ax.legend(fontsize=14)
ax.set_title("Annual Number of Blooms",fontsize=20)
ax.set_ylabel("Number of blooms per year",fontsize=14)
ax.set_xlabel("Year",fontsize=14)
ax.set_xlim(1997,2026)
ax.set_axisbelow(True)
ax.grid(axis='y',linestyle='-',alpha=0.7,color='gray')
plt.tight_layout()
plt.savefig(rf'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\Figures\Annual_Blooms_Bars',dpi=300,bbox_inches='tight')

### Bloom days per year heatmap

![Annual Events Heatmap](Figures\Annual_metrics\Bio_Annual_Blooms_Heatmap.png)

![Annual Bloom Days Biological Year](Figures\Annual_metrics\Bio_Annual_Bloom_Days_Heatmap.png)

In [ ]:
# Creates a dataframe for plotting heatmaps
data = {
    'Year': summary_MABS['Year'],
    'MAB South': summary_MABS['Percent_total_bloom_days_above_threshold_biological'],
    'MAB North': summary_MABN['Percent_total_bloom_days_above_threshold_biological'],
    'Georges Bank': summary_GB['Percent_total_bloom_days_above_threshold_biological'],
    'GOM West': summary_GOMW['Percent_total_bloom_days_above_threshold_biological'],
    'GOM East': summary_GOME['Percent_total_bloom_days_above_threshold_biological']
}
df = pd.DataFrame(data)
df = df.sort_values('Year')
data_for_heatmap = df.set_index('Year').T

In [ ]:
#Plotting a heatmap of annual events for each region (region by year)
fig, ax = plt.subplots(figsize=(10,5))
x_labels = range(1998,2026)
sparse_labels = [label if idx % 2 == 0 else "" for idx, label in enumerate(x_labels)]
num_segments = 7
original_cmap = plt.colormaps['YlGnBu']
segmented_cmap = original_cmap.resampled(num_segments)
ylabels= ['GOM\nEast','GOM\nWest','Georges\nBank','MAB\nNorth','MAB\nSouth']
ax = sns.heatmap(data_for_heatmap,
            cmap=segmented_cmap,
            annot=True,
            fmt="d",
            linewidth=0.5,
            vmin = 0,
            vmax=6,
            cbar_kws={'label':'Number of blooms per year'}
            )
cbar = ax.collections[0].colorbar
boundaries = np.linspace(0, 6, num_segments+1)
segment_centers = (boundaries[:-1] + boundaries[1:])/2
cbar.set_ticks(segment_centers)
tick_labels = ['0','1','2','3','4','5','6']
cbar.set_ticklabels(tick_labels)
ax.set_title("Annual Bloom Events",fontsize=20)
ax.set_xlabel("Biological Year",fontsize=14)
ax.set_ylabel("Region",fontsize=14)
ax.set_xticklabels(sparse_labels,fontsize=12)
ax.set_yticklabels(ylabels,fontsize=12)
plt.xticks(rotation=75)
plt.tight_layout()
plt.savefig(rf'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\Figures\Annual_Blooms_Heatmap',dpi=300,bbox_inches='tight')

In [ ]:
#Plotting annual bloom days for each region (region by year)
x_labels = ['1998/1999','1999/2000','2000/2001','2001/2002','2002/2003','2003/2004','2004/2005','2005/2006','2006/2007','2007/2008','2008/2009','2009/2010','2010/2011','2011/2012','2012/2013','2013/2014','2014/2015','2015/2016','2016/2017','2017/2018','2018/2019','2019/2020','2020/2021','2021/2022','2022/2023','2023/2024','2024/2025','2025/2026']
sparse_labels = [label if idx % 3 == 0 else "" for idx, label in enumerate(x_labels)]
fig, ax = plt.subplots(figsize=(14,5))
sns.heatmap(data_for_heatmap,
            cmap=cmocean.cm.thermal,
            annot=True,
            fmt=".2f",
            linewidth=0.5,
            cbar_kws={'label':'Number'}
            )
ax.set_title("Annual Bloom Days Above the Threshold",fontsize=20)
ax.set_xlabel("Year",fontsize=14)
ax.set_ylabel("Region",fontsize=14)
ax.set_xticklabels(sparse_labels)
plt.xticks(rotation=75)
plt.tight_layout()
#plt.savefig(rf'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\Figures\Bio_Annual_Bloom_Days_Threshold_Heatmap',dpi=300,bbox_inches='tight')

### Plotting heatmaps overlaid with Gantt charts for each region

### January - December Code

In [ ]:
def heatmap_jan_dec(summary_data,individual_data,region_title,title_fontsize=20, axes=None, colorbars='on'):
    """
    Creates a heatmap of monthly integrated chlorophyll for each year with an overlaid Gantt chart for bloom duration.

    This heatmap runs on an annual year, from January to December.

    Args: 
    """
    heatmap_x_dates = pd.to_datetime([f"2024-{str(month).zfill(2)}-01" for month in range(1,13)] + ['2025-01-01'])
    data_set = summary_data
    data_for_hm = {
        'Year': data_set['Year'],
        'January': data_set['January_integrated_chl'],
        'February': data_set['February_integrated_chl'],
        'March': data_set['March_integrated_chl'],
        'April': data_set['April_integrated_chl'],
        'May': data_set['May_integrated_chl'],
        'June': data_set['June_integrated_chl'],
        'July': data_set['July_integrated_chl'],
        'August': data_set['August_integrated_chl'],
        'September': data_set['September_integrated_chl'],
        'October': data_set['October_integrated_chl'],
        'November': data_set['November_integrated_chl'],
        'December': data_set['December_integrated_chl'],
    }
    df = pd.DataFrame(data_for_hm)
    df = df.sort_values('Year')
    heatmap_data = df.set_index('Year')

    individual_dataset = individual_data.copy()
    individual_dataset['Start date day']=pd.to_datetime(individual_dataset['Start_date'])
    individual_dataset['End date day']=pd.to_datetime(individual_dataset['End_date'])
    individual_dataset['Start_date']=individual_dataset['Start date day'].dt.strftime('%m-%d-%Y')
    individual_dataset['End_date']=individual_dataset['End date day'].dt.strftime('%m-%d-%Y')
    individual_dataset['End Year'] = individual_dataset['End date day'].dt.year

    new_dataset_rows = []

    for index, row in individual_dataset.iterrows():
        start_date = row['Start date day']
        end_date = row['End date day']
        start_year = start_date.year
        end_year = end_date.year

        current_year = start_year
        while current_year <= end_year:
            row_copy = row.copy()
            if current_year == start_year:
                row_start = start_date
            else:
                row_start = pd.Timestamp(year=current_year,month=1,day=1)
            if current_year == end_year:
                row_end = end_date
            else:
                row_end = pd.Timestamp(year=current_year,month=12,day=31,hour=23,minute=59,second=59)
            row_copy['End Year'] = current_year
            
            dummy_year = 2024
            row_copy['Normalized Start'] = pd.Timestamp(year=dummy_year,month=row_start.month,day=row_start.day)
            row_copy['Normalized End'] = pd.Timestamp(year=dummy_year,month=row_end.month,day=row_end.day)
            new_dataset_rows.append(row_copy)
            current_year = current_year+1

    individual_dataset = pd.DataFrame(new_dataset_rows).reset_index(drop=True)
    heatmap_data = heatmap_data[heatmap_data.index >= 1998]
    individual_dataset = individual_dataset[individual_dataset['End Year'] >= 1998]
    if axes is None:
        fig, ax = plt.subplots(figsize=(14,8), layout='tight')
    else:
        ax=axes
        fig = ax.get_figure()

    #Heatmap
    y_center = heatmap_data.index.astype(int).values
    y_edges = np.append(y_center - 0.5, y_center[-1] + 0.5)

    x_dates = pd.date_range(start='2024-01-01',end='2025-01-01',freq='MS')
    x_edges = mdates.date2num(x_dates)

    Z = heatmap_data.values.astype(float)
    Z[Z <= 0] = np.nan

    cmap_hm = plt.get_cmap('Greens').copy()
    cmap_hm.set_bad(color='white')
    hm = ax.pcolormesh(x_edges,y_edges,Z,cmap=cmap_hm,norm=mcolors.LogNorm(vmin=0.1,vmax=100), shading='flat')
    
    for date in heatmap_x_dates:
        ax.axvline(
            mdates.date2num(date),
            color='darkgray',
            alpha=0.5,
            linewidth=1.0,
            zorder=2
        )

    for year in y_edges:
        ax.axhline(
            year,
            color='darkgray',
            alpha=0.5,
            linewidth=1.0,
            zorder=2
        )
    lefts = mdates.date2num(individual_dataset['Normalized Start'])
    rights = mdates.date2num(individual_dataset['Normalized End'])
    widths = rights - lefts
    y_coords = individual_dataset['End Year'].astype(int)

    durations = pd.to_numeric(individual_dataset['Total_duration'], errors='coerce').fillna(0)
    norm = mcolors.Normalize(vmin=0, vmax =350)
    cmap_bars = plt.get_cmap('Wistia')
    bar_colors = cmap_bars(norm(durations))

    ax.barh(y_coords, widths, left=lefts, height =0.4, color=bar_colors, edgecolor='black', linewidth=1, zorder=5)
    #ax.set_title(f"Bloom Duration and Magnitude in {region_title}",fontsize=title_fontsize)
    ax.set_title("Calendar Year (January - December)", fontsize=18)
    ax.set_yticks(range(1998,2026))
    ylabels = ['1998','1999','2000','2001','2002','2003','2004','2005','2006','2007','2008','2009','2010','2011','2012','2013','2014','2015','2016','2017','2018','2019','2020','2021','2022','2023','2024','2025']
    sparse_labels = [label if idx % 2 == 0 else "" for idx, label in enumerate(ylabels)]
    ax.set_yticklabels(sparse_labels)
    ax.invert_yaxis()

    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
    ax.set_xlim(mdates.date2num(pd.Timestamp('2024-01-01')),mdates.date2num(pd.Timestamp('2025-01-01')))
    ax.tick_params(axis='both',labelsize=14)

    if colorbars == 'on':
        cax_hm = ax.inset_axes([1.02,0.0,0.03,1.0])
        cax_bars = ax.inset_axes([1.13,0.0,0.03,1.0])

        cbar_hm = fig.colorbar(hm, cax=cax_hm)
        cbar_hm.set_label("Integrated Chlorophyll ($mg/m^3 * days$)",fontsize=14)
        cbar_hm.ax.yaxis.set_major_formatter(ticker.ScalarFormatter())

        sm = plt.cm.ScalarMappable(cmap=cmap_bars, norm=norm)
        sm.set_array([])
        cbar_bars = fig.colorbar(sm, cax=cax_bars)
        cbar_bars.set_label("Duration of event (days)",fontsize=14)
    return fig

#### July - June Code

In [ ]:
def heatmap_jul_jun(summary_data,individual_data,region_title,title_fontsize=20, axes=None, colorbars='on'):
    heatmap_x_dates = pd.to_datetime([f"2023-{str(month).zfill(2)}-01" for month in range(7,13)] + [f"2024-{str(month).zfill(2)}-01" for month in range (1,7)])
    data_set = summary_data
    data_for_hm = {
        'Year': data_set['Year'],
        'July': data_set['July_integrated_chl'],
        'August': data_set['August_integrated_chl'],
        'September': data_set['September_integrated_chl'],
        'October': data_set['October_integrated_chl'],
        'November': data_set['November_integrated_chl'],
        'December': data_set['December_integrated_chl'],
        'January': data_set['January_integrated_chl'],
        'February': data_set['February_integrated_chl'],
        'March': data_set['March_integrated_chl'],
        'April': data_set['April_integrated_chl'],
        'May': data_set['May_integrated_chl'],
        'June': data_set['June_integrated_chl'],
    }
    df = pd.DataFrame(data_for_hm)
    df = df.sort_values('Year')
    spring_months = ['January','February','March','April','May','June']
    df[spring_months] = df[spring_months].shift(-1)
    heatmap_data = df.set_index('Year')

    individual_dataset_july_start = individual_data.copy()
    individual_dataset_july_start['Start date day']=pd.to_datetime(individual_dataset_july_start['Start_date'])
    individual_dataset_july_start['End date day']=pd.to_datetime(individual_dataset_july_start['End_date'])
    individual_dataset_july_start['Start_date']=individual_dataset_july_start['Start date day'].dt.strftime('%m-%d-%Y')
    individual_dataset_july_start['End_date']=individual_dataset_july_start['End date day'].dt.strftime('%m-%d-%Y')
    individual_dataset_july_start['End Year'] = individual_dataset_july_start['End date day'].dt.year

    new_dataset_rows = []

    for index, row in individual_dataset_july_start.iterrows():
        start_date = row['Start date day']
        end_date = row['End date day']
        if start_date.month < 7:
            start_year = start_date.year-1
        else:
            start_year = start_date.year
        if end_date.month < 7:
            end_year = end_date.year-1
        else:
            end_year = end_date.year

        current_year = start_year
        while current_year <= end_year:
            row_copy = row.copy()
            if current_year == start_year:
                row_start = start_date
            else:
                row_start = pd.Timestamp(year=current_year,month=7,day=1)
            if current_year == end_year:
                row_end = end_date
            else:
                row_end = pd.Timestamp(year=current_year+1,month=6,day=30,hour=23,minute=59,second=59)
            row_copy['End Year'] = current_year

            if row_start.month >= 7:
                dummy_start_year = 2023
            else:
                dummy_start_year = 2024
            if row_end.month >= 7:
                dummy_end_year = 2023
            else:
                dummy_end_year = 2024

            row_copy['Normalized Start'] = pd.Timestamp(year=dummy_start_year,month=row_start.month,day=row_start.day)
            row_copy['Normalized End'] = pd.Timestamp(year=dummy_end_year,month=row_end.month,day=row_end.day)
            new_dataset_rows.append(row_copy)
            current_year = current_year+1

    individual_dataset_july_start = pd.DataFrame(new_dataset_rows).reset_index(drop=True)
    heatmap_data = heatmap_data[heatmap_data.index >= 1998]
    individual_dataset_july_start = individual_dataset_july_start[individual_dataset_july_start['End Year'] >= 1998]
    if axes is None:
        fig, ax = plt.subplots(figsize=(14,8), layout='tight')
    else:
        ax = axes
        fig = ax.get_figure()

    #Heatmap
    y_center = heatmap_data.index.astype(int).values
    y_edges = np.append(y_center - 0.5, y_center[-1] + 0.5)

    x_dates = pd.date_range(start='2023-07-01',end='2024-07-01',freq='MS')
    x_edges = mdates.date2num(x_dates)

    Z = heatmap_data.values.astype(float)
    Z[Z <= 0] = np.nan

    cmap_hm = plt.get_cmap('Greens').copy()
    cmap_hm.set_bad(color='white')
    hm = ax.pcolormesh(x_edges,y_edges,Z,cmap=cmap_hm,norm=mcolors.LogNorm(vmin=0.1,vmax=100), shading='flat')
    
    for date in heatmap_x_dates:
        ax.axvline(
            mdates.date2num(date),
            color='darkgray',
            alpha=0.5,
            linewidth=1.0,
            zorder=2
        )

    for year in y_edges:
        ax.axhline(
            year,
            color='darkgray',
            alpha=0.5,
            linewidth=1.0,
            zorder=2
        )
    lefts = mdates.date2num(individual_dataset_july_start['Normalized Start'])
    rights = mdates.date2num(individual_dataset_july_start['Normalized End'])
    widths = rights - lefts
    y_coords = individual_dataset_july_start['End Year'].astype(int)

    durations = pd.to_numeric(individual_dataset_july_start['Total_duration'], errors='coerce').fillna(0)
    norm = mcolors.Normalize(vmin=0, vmax =350)
    cmap_bars = plt.get_cmap('Wistia')
    bar_colors = cmap_bars(norm(durations))

    ax.barh(y_coords, widths, left=lefts, height =0.4, color=bar_colors, edgecolor='black', linewidth=1, zorder=5)
    #ax.set_title(f"Bloom Duration and Magnitude in {str(region_title)}",fontsize=title_fontsize)
    ax.set_title("Biological Year (July - June)",fontsize=18) 
    ax.set_yticks(range(1998,2026))
    ylabels = ['1998','1999','2000','2001','2002','2003','2004','2005','2006','2007','2008','2009','2010','2011','2012','2013','2014','2015','2016','2017','2018','2019','2020','2021','2022','2023','2024','2025']
    sparse_labels = [label if idx % 2 == 0 else "" for idx, label in enumerate(ylabels)]
    ax.set_yticklabels(sparse_labels)
    ax.invert_yaxis()

    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
    ax.set_xlim(mdates.date2num(pd.Timestamp('2023-07-01')),mdates.date2num(pd.Timestamp('2024-06-30')))
    ax.tick_params(axis='both',labelsize=14)

    ax.axvline(mdates.date2num(pd.Timestamp('2024-01-01')),color='aqua',linestyle='--',linewidth=2,alpha=0.7)
    if colorbars == 'on':
        cax_hm = ax.inset_axes([1.02,0.0,0.03,1.0])
        cax_bars = ax.inset_axes([1.16,0.0,0.03,1.0])

        cbar_hm = fig.colorbar(hm, cax=cax_hm)
        cbar_hm.set_label("Integrated Chlorophyll ($mg/m^3 * days$)",fontsize=12)
        cbar_hm.ax.yaxis.set_major_formatter(ticker.ScalarFormatter())

        sm = plt.cm.ScalarMappable(cmap=cmap_bars, norm=norm)
        sm.set_array([])
        cbar_bars = fig.colorbar(sm, cax=cax_bars)
        cbar_bars.set_label("Duration of event (days)",fontsize=12)
    return fig

In [ ]:
fig, ax = plt.subplots(nrows=1,ncols=1,figsize=(15,9))
heatmap = heatmap_jul_jun(summary_GB,bloom_GB,"Georges Bank", axes=ax)
plt.savefig(r'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\Figures\GB_heatmap.png',dpi=300,bbox_inches='tight')

In [ ]:
fig, axes = plt.subplots(nrows=1,ncols=2,figsize=(18,8))
fig.subplots_adjust(wspace=0.1)
axes_flat = axes.flatten()
heatmap_jan = heatmap_jan_dec(summary_MABN,bloom_MABN,"Middle Atlantic Bight North",title_fontsize=18,axes = axes_flat[0],colorbars='off')
heatmap_jul = heatmap_jul_jun(summary_MABN,bloom_MABN,"Middle Atlantic Bight North",title_fontsize=18,axes=axes_flat[1])
fig.suptitle("Bloom Duration and Intensity in the Middle Atlantic Bight North",fontsize=20)
plt.tight_layout()
plt.savefig(r'C:\Users\grace\OneDrive\Documents\GitHub\phytoplankton_Hollings_project\Figures\Heatmaps\Heatmap_Comparison_MABN.png',dpi=300,bbox_inches='tight')

### Number of bloom classifications for each region

In [ ]:
fig, ax = plt.subplots(figsize=(18,8))
plt.grid(axis='y')
width=0.15
classifications = ['Spring', 'Fall', 'Winter', 'Summer']
x = np.arange(len(classifications))
def counts(df):
    counting = df['Bloom_classification'].value_counts()
    aligned_counts = counting.reindex(classifications,fill_value=0)
    return aligned_counts

counts_MABS = counts(bloom_MABS)
counts_MABN = counts(bloom_MABN)
counts_GB = counts(bloom_GB)
counts_GOMW = counts(bloom_GOMW)
counts_GOME = counts(bloom_GOME)

ax.bar(x-2*width,counts_MABS,color='gold',label='MAB South',edgecolor='black',width=0.15,zorder=20)
ax.bar(x-width,counts_MABN,color='cyan',label='MAB North',edgecolor='black',width=0.15,zorder=20)
ax.bar(x,counts_GB,color='darkorange',label='Georges Bank',edgecolor='black',width=0.15,zorder=20)
ax.bar(x+width,counts_GOMW,color='mediumorchid',label='GOM West',edgecolor='black',width=0.15,zorder=20)
ax.bar(x+2*width,counts_GOME,color='dodgerblue',label='GOM East',edgecolor='black',width=0.15,zorder=20)
ax.set_xticks(x)
ax.set_xticklabels(classifications)
ax.legend(fontsize=14)
ax.set_title("Classification of blooms in each region",fontsize=20)
ax.set_ylabel("Number of blooms in classification",fontsize=14)
ax.set_xlabel("Classification",fontsize=14)
plt.savefig(r'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\Figures\Annual_metrics\bloom_classification.png',dpi=300,bbox_inches='tight')

### Metric Plots of Regions

In [ ]:
def scatter_plot_data(dataset,interest_variable):
    """
    Finds the variable of interest for the spring and fall bloom.

    This function uses the metric csv file to find the variable of interest of the spring and fall blooms within a region.

    Args:
        dataset (pandas.DataFrame, required): The dataframe of all bloom metrics for the region. Must include 'Bloom ID', 'Year', and 'Bloom Classification' columns. No defaults.
        interest_variable (str, required): The variable of interest. The header in the dataframe. No defaults.
    Returns:
        tuple. A tuple containing (fall_x, fall_y, spring_x, spring_y), where:
            fall_x (list): The list of years where the fall bloom had that metric.
            fall_y (list): The list of values for the metric of interest for the fall bloom.
            spring_x (list): The list of years where the spring bloom had that metric.
            spring_y (list): The list of values for the metric of interest for the spring bloom.
    """
    fall_x = []
    fall_y = []
    spring_x = []
    spring_y = []
    variable_of_interest = str(interest_variable)

    target_values = dataset[variable_of_interest].tolist()
    years = dataset['Year'].tolist()
    bloom_classification = dataset['Bloom_classification'].tolist()
    start_DOYs = dataset['Start_DOY'].tolist()

    cross_boundary_variables = ['Peak_DOY', 'End_DOY', 'last_drop_DOY', 'First_exceed_DOY'] #Variables that might cross the year boundary

    for bloom_class, year, val, start_val in zip(bloom_classification, years, target_values, start_DOYs):
        if variable_of_interest in cross_boundary_variables:
            if not pd.isna(val) and not pd.isna(start_val) and val<start_val: #Checks for nan values and that the value is less than the start DOY value
                #Leap year verification
                is_leap_year = (year % 4 == 0 and year % 100 !=0) or (year % 400 == 0)
                val += 366 if is_leap_year else 365 #Adds 366 or 365 if the value is less than the start DOY
        if bloom_class == 'Fall':
            fall_y.append(val)
            fall_x.append(year)
        elif bloom_class == 'Spring':
            spring_y.append(val)
            spring_x.append(year)
    return fall_x, fall_y, spring_x, spring_y

In [ ]:
region_acro = ['GOME','GOMW','GB','MABN','MABS']
data = [bloom_GOME,bloom_GOMW,bloom_GB,bloom_MABN,bloom_MABS]
color_options = ['dodgerblue','mediumorchid','darkorange','cyan','gold']
axis_labels = ['DOY', 'DOY', 'DOY','Duration (days)', 'DOY', 'DOY', 'Duration (days)', 'Integrated CHlorophyll ($mg/m^3 *days)','Chlorophyll Concentration ($mg/m^3$)']
interest_variable= ['Start_DOY', 'Peak_DOY', 'End_DOY','Total_duration','First_exceed_DOY','last_drop_DOY','Duration_above_threshold','Bloom_Integrated_Chlorophyll','Maximum_chlorophyll']
variable_title= ['Initiation Day of Year', 'Peak Day of Year', 'Termination Day of Year','Duration','First Exceedance of Threshold (DOY)','Last Drop Below Threshold (DOY)','Duration Above Threshold','Bloom Integrated Chlorophyll','Maximum Chlorophyll']
titles = ['Gulf of Maine East','Gulf of Maine West','Georges Bank','Middle Atlantic Bight North','Middle Atlantic Bight South']
x_ticks = [1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025]
x_labels = ['1998','1999','2000','2001','2002','2003','2004','2005','2006','2007','2008','2009','2010','2011','2012','2013','2014','2015','2016','2017','2018','2019','2020','2021','2022','2023','2024','2025']
x_sparse_labels = [label if idx % 3 == 0 else "" for idx, label in enumerate(x_labels)]
for j in range(len(interest_variable)):
    fig_fall, axes_fall = plt.subplots(nrows=2, ncols=3, figsize=(15,9), sharey=True, sharex=True)
    axes_flat_fall = axes_fall.flatten()
    for i in range(5):
        fall_x, fall_y, spring_x, spring_y = scatter_plot_data(dataset=data[i],interest_variable=interest_variable[j])
        axes_flat_fall[i].clear()
        fall_x = np.array(fall_x)
        fall_y = np.array(fall_y)
        mask = ~np.isnan(fall_x) & ~np.isnan(fall_y)
        clean_fall_x = fall_x[mask]
        clean_fall_y = fall_y[mask]

        num_points_fall = len(clean_fall_x)
        has_enough_points_fall = num_points_fall >= 20
        sns.regplot(
            x=clean_fall_x,
            y=clean_fall_y,
            ax=axes_flat_fall[i],
            color=color_options[i],
            fit_reg=has_enough_points_fall
        )
        if 'DOY' in interest_variable[j]:
            axes_flat_fall[i].axhline(y=365, color='gray', linestyle='--', linewidth = 1, alpha=0.5)
        if has_enough_points_fall:
            slope,intercept,r_value,p_value,std_err = stats.linregress(clean_fall_x,clean_fall_y)
            r_square = r_value**2
            stats_text = f"p-value = {p_value:.4f}"
        else:
            stats_text = f"N = {num_points_fall}\n(No trendline)"
        if p_value <= 0.0559:
            axes_flat_fall[i].text(
                0.05,0.95,
                stats_text,
                transform=axes_flat_fall[i].transAxes,
                fontsize=15,
                color = 'red',
                verticalalignment='top',
                fontweight='bold'
        )
        else:
                axes_flat_fall[i].text(
                0.05,0.95,
                stats_text,
                transform=axes_flat_fall[i].transAxes,
                fontsize=15,
                color = 'black',
                verticalalignment='top'
                )
        axes_flat_fall[i].set_title(titles[i],fontsize=18)
        if i > 1:
            axes_flat_fall[i].set_xlabel('Year',fontsize=14)
        axes_flat_fall[i].set_ylabel(axis_labels[j],fontsize=13)
        axes_flat_fall[i].set_xticks(x_ticks)
        axes_flat_fall[i].set_xticklabels(x_sparse_labels,rotation=70)
        axes_flat_fall[i].tick_params(labelleft=True)
    axes_flat_fall[5].remove()

    map_projection = cartopy.crs.PlateCarree()
    ax_map = fig_fall.add_subplot(2,3,6,projection=map_projection)
    shapefile_geometry = [GOM_east_loc,GOM_west_loc,GB_whole_loc,MAB_north_loc,MAB_south_loc]
    shapefile_names = ['GOM East','GOM West','Georges Bank','MAB North','MAB South']
    black_halo = [path_effects.Stroke(linewidth=4,foreground='black'), path_effects.Normal()]
    white_halo = [path_effects.Stroke(linewidth=4,foreground='white'), path_effects.Normal()]
    MAB_south_loc.boundary.plot(ax=ax_map, color='gold', linewidth=3, path_effects=black_halo, zorder=2)
    MAB_north_loc.boundary.plot(ax=ax_map, color='cyan', linewidth=3, path_effects=black_halo, zorder=2)
    GB_whole_loc.boundary.plot(ax=ax_map, color='darkorange', linewidth=3, path_effects=black_halo, zorder=2)
    GOM_west_loc.boundary.plot(ax=ax_map, color='mediumorchid', linewidth=3, path_effects=black_halo, zorder=2)
    GOM_east_loc.boundary.plot(ax=ax_map, color='dodgerblue', linewidth=3, path_effects=black_halo, zorder=2)
    #ax_map.legend(fontsize=16,loc='lower right')
    for gdf,title,color in zip(shapefile_geometry,shapefile_names,color_options):
        for idx, row in gdf.iterrows():
            rep_point = row.geometry.representative_point()
            if title == 'MAB South':
                rot_angle = 60
                x_offset, y_offset = (16,-3)
            elif title == "Georges Bank" or title == "GOM East":
                rot_angle=0
                x_offset, y_offset = (4,-2)
            else:
                rot_angle=0
                x_offset, y_offset = (0,0)
            ax_map.annotate(text=title,
                        xy=(rep_point.x, rep_point.y),
                        xytext=(x_offset,y_offset),
                        textcoords='offset points',
                        horizontalalignment='center',
                        fontsize=8,
                        color = 'black',
                        zorder=5,
                        fontweight='bold',
                        rotation=rot_angle,
                        bbox = dict(
                            boxstyle='round,pad=0.2',
                            facecolor=color,
                            alpha=0.8,
                            linewidth=1
                        )
            )
    ax_map.add_feature(cartopy.feature.COASTLINE, linewidth=1,zorder=3)
    ax_map.add_feature(cartopy.feature.LAND, zorder=3, facecolor='darkgrey')
    ax_map.add_geometries(bathym, facecolor='none', edgecolor='black', crs=cartopy.crs.PlateCarree(),zorder=2) #Adding the shelf break line
    ax_map.set_axisbelow(False)
    ax_map.set_extent([-77,-65,35,45])
    gl = ax_map.gridlines(
        crs=cartopy.crs.PlateCarree(),
        draw_labels=True,
        color='dimgrey',
        xlocs = ticker.MultipleLocator(1.5),
        ylocs=ticker.MultipleLocator(1),
        zorder=10)
    gl.top_labels = False
    gl.right_labels = False
    ax_map.set_title('Study Locations', fontsize=20)
    axes_flat_fall[2].tick_params(labelbottom=True)
    fig_fall.suptitle("Fall Bloom " + variable_title[j] + " by Region",fontsize=20)
    filename = f"Fall_{interest_variable[j]}_scatter"
    plt.tight_layout()
    plt.savefig(rf'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\Figures\Metric_Scatter_Plots\{filename}.png',dpi=300,bbox_inches='tight')
    plt.close(fig_fall)

    fig_spring, axes_spring = plt.subplots(nrows=2, ncols=3, figsize=(15,9), sharey=True)
    axes_flat_spring = axes_spring.flatten()
    for i in range(5):
        if region_acro[i] == 'GOMW':
            data_set = data[i]
            data_set = data_set[data_set['Year'] != 2023]
        else:
            data_set = data[i]
        fall_x, fall_y, spring_x, spring_y = scatter_plot_data(dataset=data_set,interest_variable=interest_variable[j])
        axes_flat_spring[i].clear()
        spring_x = np.array(spring_x)
        spring_y = np.array(spring_y)
        mask = ~np.isnan(spring_x) & ~np.isnan(spring_y)
        clean_spring_x = spring_x[mask]
        clean_spring_y = spring_y[mask]

        num_points_spring = len(clean_spring_x)
        has_enough_points_spring = num_points_spring >= 20
        sns.regplot(
            x=clean_spring_x,
            y=clean_spring_y,
            ax=axes_flat_spring[i],
            color=color_options[i],
            fit_reg=has_enough_points_spring
        )
        if 'DOY' in interest_variable[j]:
            axes_flat_spring[i].axhline(y=365, color='gray', linestyle='--', linewidth = 1, alpha=0.5)

        if has_enough_points_spring:
            slope,intercept,r_value,p_value,std_err = stats.linregress(clean_spring_x,clean_spring_y)
            r_square = r_value**2
            stats_text = f"p-value = {p_value:.4f}"
        else:
            stats_text = f"N = {num_points_spring}\n(No trendline)"
        if p_value <=0.0559:
            axes_flat_spring[i].text(
                0.05,0.95,
                stats_text,
                transform=axes_flat_spring[i].transAxes,
                fontsize=15,
                color='red',
                verticalalignment='top',
                fontweight='bold'
            )
        else:
            axes_flat_spring[i].text(
                0.05,0.95,
                stats_text,
                transform=axes_flat_spring[i].transAxes,
                fontsize=15,
                color='black',
                verticalalignment='top'
            )
        axes_flat_spring[i].set_title(titles[i],fontsize=18)
        axes_flat_spring[i].set_xlabel('Year',fontsize=14)
        axes_flat_spring[i].set_ylabel(axis_labels[j],fontsize=13)
        axes_flat_spring[i].set_xticks(x_ticks)
        axes_flat_spring[i].set_xticklabels(x_sparse_labels,rotation=70)
        axes_flat_spring[i].tick_params(labelleft=True)
    axes_flat_spring[5].remove()

    map_projection = cartopy.crs.PlateCarree()
    ax_map_spring = fig_spring.add_subplot(2,3,6,projection=map_projection)
    shapefile_geometry = [GOM_east_loc,GOM_west_loc,GB_whole_loc,MAB_north_loc,MAB_south_loc]
    shapefile_names = ['GOM East','GOM West','Georges Bank','MAB North','MAB South']
    black_halo = [path_effects.Stroke(linewidth=4,foreground='black'), path_effects.Normal()]
    white_halo = [path_effects.Stroke(linewidth=4,foreground='white'), path_effects.Normal()]
    MAB_south_loc.boundary.plot(ax=ax_map_spring, color='gold', linewidth=3, path_effects=black_halo, zorder=2)
    MAB_north_loc.boundary.plot(ax=ax_map_spring, color='cyan', linewidth=3, path_effects=black_halo, zorder=2)
    GB_whole_loc.boundary.plot(ax=ax_map_spring, color='darkorange', linewidth=3, path_effects=black_halo, zorder=2)
    GOM_west_loc.boundary.plot(ax=ax_map_spring, color='mediumorchid', linewidth=3, path_effects=black_halo, zorder=2)
    GOM_east_loc.boundary.plot(ax=ax_map_spring, color='dodgerblue', linewidth=3, path_effects=black_halo, zorder=2)
    #ax_map_spring.legend(fontsize=16,loc='lower right')
    for gdf,title,color in zip(shapefile_geometry,shapefile_names,color_options):
        for idx, row in gdf.iterrows():
            rep_point = row.geometry.representative_point()
            if title == 'MAB South':
                rot_angle = 60
                x_offset, y_offset = (16,-3)
            elif title == "Georges Bank" or title == "GOM East":
                rot_angle=0
                x_offset, y_offset = (4,-2)
            else:
                rot_angle=0
                x_offset, y_offset = (0,0)
            ax_map_spring.annotate(text=title,
                        xy=(rep_point.x, rep_point.y),
                        xytext=(x_offset,y_offset),
                        textcoords='offset points',
                        horizontalalignment='center',
                        fontsize=8,
                        color = 'black',
                        zorder=5,
                        fontweight='bold',
                        rotation=rot_angle,
                        bbox = dict(
                            boxstyle='round,pad=0.2',
                            facecolor=color,
                            alpha=0.8,
                            linewidth=1
                        )
            )
    ax_map_spring.add_feature(cartopy.feature.COASTLINE, linewidth=1,zorder=3)
    ax_map_spring.add_feature(cartopy.feature.LAND, zorder=3, facecolor='darkgrey')
    ax_map_spring.add_geometries(bathym, facecolor='none', edgecolor='black', crs=cartopy.crs.PlateCarree(),zorder=2) #Adding the shelf break line
    ax_map_spring.set_axisbelow(False)
    ax_map_spring.set_extent([-77,-65,35,45])
    gl = ax_map_spring.gridlines(
        crs=cartopy.crs.PlateCarree(),
        draw_labels=True,
        color='dimgrey',
        xlocs = ticker.MultipleLocator(1.5),
        ylocs=ticker.MultipleLocator(1),
        zorder=10)
    gl.top_labels = False
    gl.right_labels = False
    ax_map_spring.set_title('Study Locations', fontsize=20)
    fig_spring.suptitle("Spring Bloom " + variable_title[j] + " by Region",fontsize=20)
    filename = f"Spring_{interest_variable[j]}_scatter_no2023"
    plt.tight_layout()
    plt.savefig(rf'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\Figures\Metric_Scatter_Plots\{filename}.png',dpi=300,bbox_inches='tight')
    plt.close(fig_spring)

### Region Plots of Metrics

In [ ]:
region_acro = ['MABS','MABN','GB','GOMW','GOME']
data = [bloom_MABS,bloom_MABN,bloom_GB,bloom_GOMW,bloom_GOME]
color_options = ['gold','mediumturquoise','darkorange','mediumorchid','dodgerblue']
interest_variable= ['Start_DOY', 'Peak_DOY', 'End_DOY','Total_duration','Bloom_Integrated_Chlorophyll','Maximum_chlorophyll']
variable_axis = ['DOY', '', '', 'Duration (Days)', 'Integrated Chlorophyll ($mg/m^3 * days$)','Chlorophyll concentration ($mg/m^3$)' ]
variable_title= ['Bloom Initiation', 'Bloom Peak', 'Bloom Termination','Duration','Integrated Chlorophyll','Maximum Chlorophyll']
titles = ['Middle Atlantic Bight South','Middle Atlantic Bight North','Georges Bank','Gulf of Maine West','Gulf of Maine East']
x_ticks = [1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025]
x_labels = ['1998','1999','2000','2001','2002','2003','2004','2005','2006','2007','2008','2009','2010','2011','2012','2013','2014','2015','2016','2017','2018','2019','2020','2021','2022','2023','2024','2025']
sparse_labels = [label if idx % 3 == 0 else "" for idx, label in enumerate(x_labels)]
for i in range(5):
    fig_fall, axes_fall = plt.subplots(nrows=2, ncols=3, figsize=(15,9),sharex=True)
    axes_flat_fall = axes_fall.flatten()
    dataset = data[i]
    for j in range(len(interest_variable)):
        fall_x, fall_y, spring_x, spring_y = scatter_plot_data(dataset=dataset,interest_variable=interest_variable[j])
        sns.regplot(
            x=fall_x,
            y=fall_y,
            ax=axes_flat_fall[j],
            color=color_options[i]
        )
        fall_x = np.array(fall_x)
        fall_y = np.array(fall_y)
        mask = ~np.isnan(fall_x) & ~np.isnan(fall_y)
        clean_fall_x = fall_x[mask]
        clean_fall_y = fall_y[mask]
        slope,intercept,r_value,p_value,std_err = stats.linregress(clean_fall_x,clean_fall_y)
        stats_text = f"p-value = {p_value:.4f}"
        if p_value <= 0.0559:
            axes_flat_fall[j].text(
                0.05,0.95,
                stats_text,
                transform=axes_flat_fall[j].transAxes,
                fontsize=15,
                color = 'red',
                verticalalignment='top',
                fontweight='bold'
        )
        else:
                axes_flat_fall[j].text(
                0.05,0.95,
                stats_text,
                transform=axes_flat_fall[j].transAxes,
                fontsize=15,
                color = 'black',
                verticalalignment='top'
                )
        axes_flat_fall[j].set_title(variable_title[j],fontsize=18)
        axes_flat_fall[j].set_ylabel(variable_axis[j],fontsize=13)
        axes_flat_fall[j].set_xticks(x_ticks)
        axes_flat_fall[j].set_xticklabels(sparse_labels,rotation=70)
        axes_flat_fall[j].tick_params(axis = 'both',labelsize=13)
        if j > 2:
            axes_flat_fall[j].set_xlabel('Year',fontsize=14)
        elif j <= 2:
            axes_flat_fall[j].set_ylim(200,525)
        if 'DOY' in interest_variable[j]:
            axes_flat_fall[j].axhline(y=365, color='gray', linestyle='--', linewidth = 1, alpha=0.5)
    fig_fall.suptitle(f"{titles[i]} Fall Bloom Metrics",fontsize=20)
    filename = f"{region_acro[i]}_Fall_Bloom_Metrics_scatter"
    plt.tight_layout()
    plt.savefig(rf'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\Figures\Regional_scatter_plots\{filename}.png',dpi=300,bbox_inches='tight')
    plt.close(fig_fall)

    fig_spring, axes_spring = plt.subplots(nrows=2, ncols=3, figsize=(15,9))
    axes_flat_spring = axes_spring.flatten()
    """ if region_acro[i] == 'GOMW':
        data_set = data[i]
        data_set = data_set[data_set['Year'] != 2023]
    else:
        data_set = data[i]"""
    data_set = data[i]
    for j in range(len(interest_variable)):
        fall_x, fall_y, spring_x, spring_y = scatter_plot_data(dataset=data_set,interest_variable=interest_variable[j])
        sns.regplot(
            x=spring_x,
            y=spring_y,
            ax=axes_flat_spring[j],
            color=color_options[i]
        )
        spring_x = np.array(spring_x)
        spring_y = np.array(spring_y)
        mask = ~np.isnan(spring_x) & ~np.isnan(spring_y)
        clean_spring_x = spring_x[mask]
        clean_spring_y = spring_y[mask]
        slope,intercept,r_value,p_value,std_err = stats.linregress(clean_spring_x,clean_spring_y)
        r_square = r_value**2
        stats_text = f"p-value = {p_value:.4f}"
        if p_value <=0.0559:
            axes_flat_spring[j].text(
                0.05,0.95,
                stats_text,
                transform=axes_flat_spring[j].transAxes,
                fontsize=15,
                color='red',
                verticalalignment='top',
                fontweight='bold'
            )
        else:
            axes_flat_spring[j].text(
                0.05,0.95,
                stats_text,
                transform=axes_flat_spring[j].transAxes,
                fontsize=15,
                color='black',
                verticalalignment='top'
            )
        axes_flat_spring[j].set_title(variable_title[j],fontsize=18)
        axes_flat_spring[j].set_xlabel('Year',fontsize=14)
        axes_flat_spring[j].set_ylabel(variable_axis[j],fontsize=13)
        axes_flat_spring[j].set_xticks(x_ticks)
        axes_flat_spring[j].set_xticklabels(sparse_labels,rotation=70)
        axes_flat_spring[j].tick_params(axis='both', labelsize=13)
        if 'DOY' in interest_variable[j]:
            axes_flat_spring[j].axhline(y=365, color='gray', linestyle='--', linewidth = 1, alpha=0.5)
    fig_spring.suptitle(f"{titles[i]} Spring Bloom Metrics",fontsize=20)
    """if region_acro[i] == 'GOMW':
        filename = f"{region_acro[i]}_Spring_Bloom_Metrics_scatter_no2023"
        print("This is the GOMW")
    else:"""
    filename = f"{region_acro[i]}_Spring_Bloom_Metrics_scatter"
    plt.tight_layout()
    plt.savefig(rf'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\Figures\Regional_scatter_plots\{filename}.png',dpi=300,bbox_inches='tight')
    plt.close(fig_spring)

#### Fall and Spring Metrics

In [ ]:
region_acro = ['GB','GOMW','GOME']
data = [bloom_GB, bloom_GOMW, bloom_GOME]
color_options = ['darkorange','mediumorchid','dodgerblue']
interest_variable= ['Start_DOY', 'End_DOY', 'Total_duration']
variable_title= ['Initiation', 'Termination', 'Duration']
axis_labels = ['DOY', 'DOY', 'Days']
titles = ['Georges Bank','Gulf of Maine West','Gulf of Maine East']
x_ticks = [1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025]
x_labels = ['1998','1999','2000','2001','2002','2003','2004','2005','2006','2007','2008','2009','2010','2011','2012','2013','2014','2015','2016','2017','2018','2019','2020','2021','2022','2023','2024','2025']
sparse_labels = [label if idx % 3 == 0 else "" for idx, label in enumerate(x_labels)]
for i in range(3):
    fig, axes = plt.subplots(nrows=2,ncols=3,figsize=(18,9),sharex=True)
    axes_flat = axes.flatten()
    for j in range(len(interest_variable)):
        if region_acro[i] == 'GOMW' or region_acro[i] == 'GOME':
            data_set = data[i]
            data_set = data_set[data_set['Year'] != 2023]
        else:
            data_set = data[i]
        fall_x, fall_y, spring_x, spring_y = scatter_plot_data(dataset=data_set,interest_variable=interest_variable[j])
        #Spring
        ax_spring = axes_flat[j]
        sns.regplot(
            x=spring_x,
            y=spring_y,
            ax=ax_spring,
            color='forestgreen',
        )
        spring_x = np.array(spring_x)
        spring_y = np.array(spring_y)
        mask = ~np.isnan(spring_x) & ~np.isnan(spring_y)
        clean_spring_x = spring_x[mask]
        clean_spring_y = spring_y[mask]
        slope,intercept,r_value,p_value,std_err = stats.linregress(clean_spring_x,clean_spring_y)
        r_square = r_value**2
        stats_text = f"p-value = {p_value:.4f}"
        if p_value <=0.0559:
            ax_spring.text(
                0.05,0.95,
                stats_text,
                transform=ax_spring.transAxes,
                fontsize=15,
                color='red',
                verticalalignment='top',
                fontweight='bold'
            )
        else:
            ax_spring.text(
                0.05,0.95,
                stats_text,
                transform=ax_spring.transAxes,
                fontsize=15,
                color='black',
                verticalalignment='top'
            )
        ax_spring.set_title(variable_title[j],fontsize=18)
        ax_spring.set_xlabel('Year',fontsize=14)
        ax_spring.set_xticks(x_ticks)
        ax_spring.set_xticklabels(sparse_labels,rotation=70)
        ax_spring.tick_params(labelsize=12)
        if "DOY" in interest_variable[j]:
            ax_spring.set_ylim(25,200)
            ax_spring.axhline(y=365, color='gray', linestyle='--', linewidth = 1, alpha=0.5)

        #Fall
        fall_x, fall_y, spring_x, spring_y = scatter_plot_data(dataset=data[i],interest_variable=interest_variable[j])
        ax_fall = axes_flat[j+3]
        sns.regplot(
            x=fall_x,
            y=fall_y,
            ax=ax_fall,
            color='darkorange',
        )
        fall_x = np.array(fall_x)
        fall_y = np.array(fall_y)
        mask = ~np.isnan(fall_x) & ~np.isnan(fall_y)
        clean_fall_x = fall_x[mask]
        clean_fall_y = fall_y[mask]
        slope,intercept,r_value,p_value,std_err = stats.linregress(clean_fall_x,clean_fall_y)
        r_square = r_value**2
        stats_text = f"p-value = {p_value:.4f}"
        if p_value <=0.0559:
            ax_fall.text(
                0.05,0.95,
                stats_text,
                transform=ax_fall.transAxes,
                fontsize=15,
                color='red',
                verticalalignment='top',
                fontweight='bold'
            )
        else:
            ax_fall.text(
                0.05,0.95,
                stats_text,
                transform=ax_fall.transAxes,
                fontsize=15,
                color='black',
                verticalalignment='top'
            )
        ax_fall.set_title(variable_title[j],fontsize=18)
        ax_fall.set_xticks(x_ticks)
        ax_fall.set_xticklabels(sparse_labels,rotation=70)
        ax_fall.tick_params(labelsize=12)
        if 'DOY' in interest_variable[j]:
            ax_fall.set_ylim(220,440)
            ax_fall.axhline(y=365, color='gray', linestyle='--', linewidth = 1, alpha=0.5)

    fig.suptitle(f"{titles[i]} Fall and Spring Bloom Metrics",fontsize=30)
    fig.text(0.09,0.50,"Day of Year (DOY)",fontsize=18,va='center',ha='center',rotation='vertical')
    fig.text(0.07, 0.72, 'Spring Bloom', fontsize=18, fontweight='bold', va='center', ha='center', rotation='vertical')
    fig.text(0.07, 0.26, 'Fall Bloom', fontsize=18, fontweight='bold', va='center', ha='center', rotation='vertical')

    filename = f"{region_acro[i]}_Fall_Spring_Bloom_Metrics_scatter_no2023"
    #plt.tight_layout()
    plt.savefig(rf'C:\Users\grace\OneDrive\Documents\GitHub\phytoplankton_Hollings_project\Figures\Regional_scatter_plots\{filename}.png',dpi=300,bbox_inches='tight')
    plt.close(fig)

### Integrated Chlorophyll Bar Charts

In [ ]:
titles = ['Gulf of Maine East','Gulf of Maine West','Georges Bank','Middle Atlantic Bight North','Middle Atlantic Bight South']
MABS_pivot = bloom_MABS.pivot_table(
    values='Bloom_Integrated_Chlorophyll',
    index='Year',
    columns='Bloom_classification',
    aggfunc='sum',
    fill_value=0
)
MABN_pivot = bloom_MABN.pivot_table(
    values='Bloom_Integrated_Chlorophyll',
    index='Year',
    columns='Bloom_classification',
    aggfunc='sum',
    fill_value=0
)
GB_pivot = bloom_GB.pivot_table(
    values='Bloom_Integrated_Chlorophyll',
    index='Year',
    columns='Bloom_classification',
    aggfunc='sum',
    fill_value=0
)
GOMW_pivot = bloom_GOMW.pivot_table(
    values='Bloom_Integrated_Chlorophyll',
    index='Year',
    columns='Bloom_classification',
    aggfunc='sum',
    fill_value=0
)
GOME_pivot = bloom_GOME.pivot_table(
    values='Bloom_Integrated_Chlorophyll',
    index='Year',
    columns='Bloom_classification',
    aggfunc='sum',
    fill_value=0
)
stack_order = ['Fall','Summer','Spring','Winter']
color_map = {
    "Fall": "darkorange",
    "Summer": "deeppink",
    "Spring": "darkorchid",
    "Winter": "mediumblue",
}
stack_colors = [color_map[col] for col in stack_order]
total_lists = [summary_GOME,summary_GOMW,summary_GB,summary_MABN,summary_MABS]
pivot_lists = [GOME_pivot,GOMW_pivot,GB_pivot,MABN_pivot,MABS_pivot]
fig, axes = plt.subplots(nrows=2,ncols=3,figsize=(14,8),sharey=True, sharex=True)
axes_flat = axes.flatten()
all_years = list(range(1998,2026))
for i in range(5):
    ax = axes_flat[i]
    df_summary = total_lists[i].copy()
    if 'Year' in df_summary.columns:
        df_summary = df_summary.set_index('Year')
    df_summary.index = df_summary.index.astype(int)

    clean_total_series = df_summary['Biological_total_integrated'].reindex(all_years,fill_value=0)
    clean_pivot = pivot_lists[i].reindex(index=all_years, columns=stack_order, fill_value=0)
    clean_total_series.plot(kind='bar',width=0.8,ax=ax,color='mediumseagreen',label='Non-Bloom Chlorophyll')
    clean_pivot.plot(kind='bar',stacked=True,width=0.8,ax=ax,legend=False,color=stack_colors)
    ax.set_xlabel('Biological Year',fontsize=14)
    ax.tick_params(labelleft=True)
    ax.tick_params(labelbottom=True)
    ax.set_xticks(range(len(all_years)))
    xticks = [str(int(x)) for x in range(1998,2026)]
    sparse_labels = [label if idx % 2 == 0 else "" for idx, label in enumerate(xticks)]
    ax.set_xticklabels(sparse_labels,rotation=80,fontsize=13)
    ax.set_ylim(0,950)
    ax.set_title(titles[i],fontsize=18)

axes_flat[5].axis('off')
axes_flat[1].tick_params(labelleft=True)
axes_flat[2].tick_params(labelleft=True)
axes_flat[1].tick_params(labelbottom=True)
axes_flat[2].tick_params(labelbottom=True)
axes_flat[0].tick_params(labelbottom=True)
handles,labels = axes_flat[0].get_legend_handles_labels()

label_handle_map = dict(zip(labels,handles))
label_order = [
    'Non-Bloom Chlorophyll',
    'Winter',
    'Spring',
    'Summer',
    'Fall'
]
labels_renamed = {
    'Non-Bloom Chlorophyll': 'Non-Bloom Chlorophyll',
    'Winter':'Winter (Jan-Feb)',
    'Spring': 'Spring (Mar-May)',
    'Summer': 'Summer (Jun-Aug)',
    'Fall': 'Fall (Sept-Dec)'
}
ordered_handles = [label_handle_map[lbl] for lbl in label_order if lbl in label_handle_map]
ordered_labels = [labels_renamed.get(lbl, lbl) for lbl in label_order if lbl in label_handle_map]
axes_flat[5].legend(ordered_handles,ordered_labels,title='Bloom Classification',title_fontsize=16,fontsize=15,loc='center')
plt.suptitle("Regional Integrated Chlorophyll",fontsize=20)
fig.supylabel("Integrated Chlorophyll Concentration ($mg/m^3 *days$)",fontsize=16)
plt.tight_layout()
plt.savefig(r'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\Figures\Annual_metrics\region_int_chl_bio_year.png',dpi=300,bbox_inches='tight')

In [ ]:
region_acro = ['GOME','GOMW','GB','MABN','MABS']
data = [bloom_GOME,bloom_GOMW,bloom_GB,bloom_MABN,bloom_MABS]
color_options = ['dodgerblue','mediumorchid','darkorange','cyan','gold']
axis_labels = ['DOY', 'DOY', 'DOY', 'DOY', 'DOY']
interest_variable= ['Days_between_initiations','Days_between_terminations']
variable_title= ['Days Between ROC and thld Start', 'Days between ROC and thld end']
titles = ['Gulf of Maine East','Gulf of Maine West','Georges Bank','Middle Atlantic Bight North','Middle Atlantic Bight South']
x_ticks = [1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025]
x_labels = ['1998','1999','2000','2001','2002','2003','2004','2005','2006','2007','2008','2009','2010','2011','2012','2013','2014','2015','2016','2017','2018','2019','2020','2021','2022','2023','2024','2025']
x_sparse_labels = [label if idx % 3 == 0 else "" for idx, label in enumerate(x_labels)]
for j in range(len(interest_variable)):
    fig_fall, axes_fall = plt.subplots(nrows=2, ncols=3, figsize=(15,9), sharey=True, sharex=True)
    axes_flat_fall = axes_fall.flatten()
    for i in range(5):
        fall_x, fall_y, spring_x, spring_y = scatter_plot_data(dataset=data[i],interest_variable=interest_variable[j])
        axes_flat_fall[i].clear()
        fall_x = np.array(fall_x)
        fall_y = np.array(fall_y)
        mask = ~np.isnan(fall_x) & ~np.isnan(fall_y)
        clean_fall_x = fall_x[mask]
        clean_fall_y = fall_y[mask]

        num_points_fall = len(clean_fall_x)
        has_enough_points_fall = num_points_fall >= 20
        sns.regplot(
            x=clean_fall_x,
            y=clean_fall_y,
            ax=axes_flat_fall[i],
            color=color_options[i],
            fit_reg=has_enough_points_fall
        )
        if 'DOY' in interest_variable[j]:
            axes_flat_fall[i].axhline(y=365, color='gray', linestyle='--', linewidth = 1, alpha=0.5)
        if has_enough_points_fall:
            slope,intercept,r_value,p_value,std_err = stats.linregress(clean_fall_x,clean_fall_y)
            r_square = r_value**2
            stats_text = f"$r^2$ value = {r_square:.4f}\np-value = {p_value:.4f}"
        else:
            stats_text = f"N = {num_points_fall}\n(No trendline)"
        if p_value <= 0.0559:
            axes_flat_fall[i].text(
                0.05,0.95,
                stats_text,
                transform=axes_flat_fall[i].transAxes,
                fontsize=15,
                color = 'red',
                verticalalignment='top',
                fontweight='bold'
        )
        else:
                axes_flat_fall[i].text(
                0.05,0.95,
                stats_text,
                transform=axes_flat_fall[i].transAxes,
                fontsize=15,
                color = 'black',
                verticalalignment='top'
                )
        axes_flat_fall[i].set_title(titles[i],fontsize=18)
        if i > 1:
            axes_flat_fall[i].set_xlabel('Year',fontsize=14)
        axes_flat_fall[i].set_ylabel(axis_labels[j],fontsize=13)
        axes_flat_fall[i].set_xticks(x_ticks)
        axes_flat_fall[i].set_xticklabels(x_sparse_labels,rotation=70)
        axes_flat_fall[i].tick_params(labelleft=True)
    axes_flat_fall[5].remove()

    map_projection = cartopy.crs.PlateCarree()
    ax_map = fig_fall.add_subplot(2,3,6,projection=map_projection)
    shapefile_geometry = [GOM_east_loc,GOM_west_loc,GB_whole_loc,MAB_north_loc,MAB_south_loc]
    shapefile_names = ['GOM East','GOM West','Georges Bank','MAB North','MAB South']
    black_halo = [path_effects.Stroke(linewidth=4,foreground='black'), path_effects.Normal()]
    white_halo = [path_effects.Stroke(linewidth=4,foreground='white'), path_effects.Normal()]
    MAB_south_loc.boundary.plot(ax=ax_map, color='gold', linewidth=3, path_effects=black_halo, zorder=2)
    MAB_north_loc.boundary.plot(ax=ax_map, color='cyan', linewidth=3, path_effects=black_halo, zorder=2)
    GB_whole_loc.boundary.plot(ax=ax_map, color='darkorange', linewidth=3, path_effects=black_halo, zorder=2)
    GOM_west_loc.boundary.plot(ax=ax_map, color='mediumorchid', linewidth=3, path_effects=black_halo, zorder=2)
    GOM_east_loc.boundary.plot(ax=ax_map, color='dodgerblue', linewidth=3, path_effects=black_halo, zorder=2)
    #ax_map.legend(fontsize=16,loc='lower right')
    for gdf,title,color in zip(shapefile_geometry,shapefile_names,color_options):
        for idx, row in gdf.iterrows():
            rep_point = row.geometry.representative_point()
            if title == 'MAB South':
                rot_angle = 60
                x_offset, y_offset = (16,-3)
            elif title == "Georges Bank" or title == "GOM East":
                rot_angle=0
                x_offset, y_offset = (4,-2)
            else:
                rot_angle=0
                x_offset, y_offset = (0,0)
            ax_map.annotate(text=title,
                        xy=(rep_point.x, rep_point.y),
                        xytext=(x_offset,y_offset),
                        textcoords='offset points',
                        horizontalalignment='center',
                        fontsize=8,
                        color = 'black',
                        zorder=5,
                        fontweight='bold',
                        rotation=rot_angle,
                        bbox = dict(
                            boxstyle='round,pad=0.2',
                            facecolor=color,
                            alpha=0.8,
                            linewidth=1
                        )
            )
    ax_map.add_feature(cartopy.feature.COASTLINE, linewidth=1,zorder=3)
    ax_map.add_feature(cartopy.feature.LAND, zorder=3, facecolor='darkgrey')
    ax_map.add_geometries(bathym, facecolor='none', edgecolor='black', crs=cartopy.crs.PlateCarree(),zorder=2) #Adding the shelf break line
    ax_map.set_axisbelow(False)
    ax_map.set_extent([-77,-65,35,45])
    gl = ax_map.gridlines(
        crs=cartopy.crs.PlateCarree(),
        draw_labels=True,
        color='dimgrey',
        xlocs = ticker.MultipleLocator(1.5),
        ylocs=ticker.MultipleLocator(1),
        zorder=10)
    gl.top_labels = False
    gl.right_labels = False
    ax_map.set_title('Study Locations', fontsize=20)
    axes_flat_fall[2].tick_params(labelbottom=True)
    fig_fall.suptitle("Fall Bloom " + variable_title[j] + " by Region",fontsize=20)
    filename = f"Fall_{interest_variable[j]}_scatter"
    plt.tight_layout()
    plt.savefig(rf'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\Figures\Metric_Scatter_Plots\{filename}.png',dpi=300,bbox_inches='tight')
    plt.close(fig_fall)

    fig_spring, axes_spring = plt.subplots(nrows=2, ncols=3, figsize=(15,9), sharey=True)
    axes_flat_spring = axes_spring.flatten()
    for i in range(5):
        #if region_acro[i] == 'GOMW':
            #data_set = data[i]
            #data_set = data_set[data_set['Year'] != 2023]
        #else:
            #data_set = data[i]
        data_set = data[i]
        fall_x, fall_y, spring_x, spring_y = scatter_plot_data(dataset=data_set,interest_variable=interest_variable[j])
        axes_flat_spring[i].clear()
        spring_x = np.array(spring_x)
        spring_y = np.array(spring_y)
        mask = ~np.isnan(spring_x) & ~np.isnan(spring_y)
        clean_spring_x = spring_x[mask]
        clean_spring_y = spring_y[mask]

        num_points_spring = len(clean_spring_x)
        has_enough_points_spring = num_points_spring >= 20
        sns.regplot(
            x=clean_spring_x,
            y=clean_spring_y,
            ax=axes_flat_spring[i],
            color=color_options[i],
            fit_reg=has_enough_points_spring
        )
        if 'DOY' in interest_variable[j]:
            axes_flat_spring[i].axhline(y=365, color='gray', linestyle='--', linewidth = 1, alpha=0.5)

        if has_enough_points_spring:
            slope,intercept,r_value,p_value,std_err = stats.linregress(clean_spring_x,clean_spring_y)
            r_square = r_value**2
            stats_text = f"$r^2$ value = {r_square:.4f}\np-value = {p_value:.4f}"
        else:
            stats_text = f"N = {num_points_spring}\n(No trendline)"
        if p_value <=0.0559:
            axes_flat_spring[i].text(
                0.05,0.95,
                stats_text,
                transform=axes_flat_spring[i].transAxes,
                fontsize=15,
                color='red',
                verticalalignment='top',
                fontweight='bold'
            )
        else:
            axes_flat_spring[i].text(
                0.05,0.95,
                stats_text,
                transform=axes_flat_spring[i].transAxes,
                fontsize=15,
                color='black',
                verticalalignment='top'
            )
        axes_flat_spring[i].set_title(titles[i],fontsize=18)
        axes_flat_spring[i].set_xlabel('Year',fontsize=14)
        axes_flat_spring[i].set_ylabel(axis_labels[j],fontsize=13)
        axes_flat_spring[i].set_xticks(x_ticks)
        axes_flat_spring[i].set_xticklabels(x_sparse_labels,rotation=70)
        axes_flat_spring[i].tick_params(labelleft=True)
    axes_flat_spring[5].remove()

    map_projection = cartopy.crs.PlateCarree()
    ax_map_spring = fig_spring.add_subplot(2,3,6,projection=map_projection)
    shapefile_geometry = [GOM_east_loc,GOM_west_loc,GB_whole_loc,MAB_north_loc,MAB_south_loc]
    shapefile_names = ['GOM East','GOM West','Georges Bank','MAB North','MAB South']
    black_halo = [path_effects.Stroke(linewidth=4,foreground='black'), path_effects.Normal()]
    white_halo = [path_effects.Stroke(linewidth=4,foreground='white'), path_effects.Normal()]
    MAB_south_loc.boundary.plot(ax=ax_map_spring, color='gold', linewidth=3, path_effects=black_halo, zorder=2)
    MAB_north_loc.boundary.plot(ax=ax_map_spring, color='cyan', linewidth=3, path_effects=black_halo, zorder=2)
    GB_whole_loc.boundary.plot(ax=ax_map_spring, color='darkorange', linewidth=3, path_effects=black_halo, zorder=2)
    GOM_west_loc.boundary.plot(ax=ax_map_spring, color='mediumorchid', linewidth=3, path_effects=black_halo, zorder=2)
    GOM_east_loc.boundary.plot(ax=ax_map_spring, color='dodgerblue', linewidth=3, path_effects=black_halo, zorder=2)
    #ax_map_spring.legend(fontsize=16,loc='lower right')
    for gdf,title,color in zip(shapefile_geometry,shapefile_names,color_options):
        for idx, row in gdf.iterrows():
            rep_point = row.geometry.representative_point()
            if title == 'MAB South':
                rot_angle = 60
                x_offset, y_offset = (16,-3)
            elif title == "Georges Bank" or title == "GOM East":
                rot_angle=0
                x_offset, y_offset = (4,-2)
            else:
                rot_angle=0
                x_offset, y_offset = (0,0)
            ax_map_spring.annotate(text=title,
                        xy=(rep_point.x, rep_point.y),
                        xytext=(x_offset,y_offset),
                        textcoords='offset points',
                        horizontalalignment='center',
                        fontsize=8,
                        color = 'black',
                        zorder=5,
                        fontweight='bold',
                        rotation=rot_angle,
                        bbox = dict(
                            boxstyle='round,pad=0.2',
                            facecolor=color,
                            alpha=0.8,
                            linewidth=1
                        )
            )
    ax_map_spring.add_feature(cartopy.feature.COASTLINE, linewidth=1,zorder=3)
    ax_map_spring.add_feature(cartopy.feature.LAND, zorder=3, facecolor='darkgrey')
    ax_map_spring.add_geometries(bathym, facecolor='none', edgecolor='black', crs=cartopy.crs.PlateCarree(),zorder=2) #Adding the shelf break line
    ax_map_spring.set_axisbelow(False)
    ax_map_spring.set_extent([-77,-65,35,45])
    gl = ax_map_spring.gridlines(
        crs=cartopy.crs.PlateCarree(),
        draw_labels=True,
        color='dimgrey',
        xlocs = ticker.MultipleLocator(1.5),
        ylocs=ticker.MultipleLocator(1),
        zorder=10)
    gl.top_labels = False
    gl.right_labels = False
    ax_map_spring.set_title('Study Locations', fontsize=20)
    fig_spring.suptitle("Spring Bloom " + variable_title[j] + " by Region",fontsize=20)
    filename = f"Spring_{interest_variable[j]}_scatter"
    plt.tight_layout()
    plt.savefig(rf'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\Figures\Metric_Scatter_Plots\{filename}.png',dpi=300,bbox_inches='tight')
    plt.close(fig_spring)

### Testing threshold start DOY vs ROC start DOY

In [ ]:
x_data = bloom_MABS['Bloom_Integrated_Chlorophyll']
y_data = bloom_MABS['Bloom_Integrated_Chlorophyll_Above_Threshold']
fig, axes = plt.subplots(nrows=1,ncols=1)
sns.regplot(
    x= x_data,
    y= y_data
)
slope,intercept,r_value,p_value,std_err = stats.linregress(x_data,y_data)
r_square = r_value**2
stats_text = f"$r^2$ value = {r_square:.4f}\np-value = {p_value:.4f}"
if p_value <= 0.0559:
    axes.text(
        0.05,0.95,
        stats_text,
        transform=axes.transAxes,
        fontsize=15,
        color = 'red',
        verticalalignment='top',
        fontweight='bold'
)
else:
        axes.text(
        0.05,0.95,
        stats_text,
        transform=axes.transAxes,
        fontsize=15,
        color = 'black',
        verticalalignment='top'
        )

In [ ]:
region_acro = ['GOME','GOMW','GB','MABN','MABS']
data = [bloom_GOME,bloom_GOMW,bloom_GB,bloom_MABN,bloom_MABS]
color_options = ['dodgerblue','mediumorchid','darkorange','cyan','gold']
axis_labels = ['Number of peaks']
interest_variable= ['Number_of_peaks']
variable_title= ['Number of Peaks In Each Event']
titles = ['Gulf of Maine East','Gulf of Maine West','Georges Bank','Middle Atlantic Bight North','Middle Atlantic Bight South']
x_ticks = [1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025]
x_labels = ['1998','1999','2000','2001','2002','2003','2004','2005','2006','2007','2008','2009','2010','2011','2012','2013','2014','2015','2016','2017','2018','2019','2020','2021','2022','2023','2024','2025']
x_sparse_labels = [label if idx % 3 == 0 else "" for idx, label in enumerate(x_labels)]
for j in range(len(interest_variable)):
    fig_fall, axes_fall = plt.subplots(nrows=2, ncols=3, figsize=(15,9), sharey=True, sharex=True)
    axes_flat_fall = axes_fall.flatten()
    for i in range(5):
        fall_x, fall_y, spring_x, spring_y = scatter_plot_data(dataset=data[i],interest_variable=interest_variable[j])
        axes_flat_fall[i].clear()
        fall_x = np.array(fall_x)
        fall_y = np.array(fall_y)
        mask = ~np.isnan(fall_x) & ~np.isnan(fall_y)
        clean_fall_x = fall_x[mask]
        clean_fall_y = fall_y[mask]

        num_points_fall = len(clean_fall_x)
        has_enough_points_fall = num_points_fall >= 20
        sns.regplot(
            x=clean_fall_x,
            y=clean_fall_y,
            ax=axes_flat_fall[i],
            color=color_options[i],
            fit_reg=has_enough_points_fall
        )
        if 'DOY' in interest_variable[j]:
            axes_flat_fall[i].axhline(y=365, color='gray', linestyle='--', linewidth = 1, alpha=0.5)
        if has_enough_points_fall:
            slope,intercept,r_value,p_value,std_err = stats.linregress(clean_fall_x,clean_fall_y)
            r_square = r_value**2
            stats_text = f"p-value = {p_value:.4f}"
        else:
            stats_text = f"N = {num_points_fall}\n(No trendline)"
        if p_value <= 0.0559:
            axes_flat_fall[i].text(
                0.05,0.95,
                stats_text,
                transform=axes_flat_fall[i].transAxes,
                fontsize=15,
                color = 'red',
                verticalalignment='top',
                fontweight='bold'
        )
        else:
                axes_flat_fall[i].text(
                0.05,0.95,
                stats_text,
                transform=axes_flat_fall[i].transAxes,
                fontsize=15,
                color = 'black',
                verticalalignment='top'
                )
        axes_flat_fall[i].set_title(titles[i],fontsize=18)
        if i > 1:
            axes_flat_fall[i].set_xlabel('Year',fontsize=14)
        axes_flat_fall[i].set_ylabel(axis_labels[j],fontsize=13)
        axes_flat_fall[i].set_xticks(x_ticks)
        axes_flat_fall[i].set_xticklabels(x_sparse_labels,rotation=70)
        axes_flat_fall[i].tick_params(labelleft=True)
    axes_flat_fall[5].remove()

    axes_flat_fall[2].tick_params(labelbottom=True)
    fig_fall.suptitle("Fall Bloom " + variable_title[j] + " by Region",fontsize=20)
    filename = f"Fall_{interest_variable[j]}_scatter"
    plt.tight_layout()
    plt.savefig(rf'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\Figures\Metric_Scatter_Plots\{filename}.png',dpi=300,bbox_inches='tight')
    plt.close(fig_fall)

    fig_spring, axes_spring = plt.subplots(nrows=2, ncols=3, figsize=(15,9), sharey=True)
    axes_flat_spring = axes_spring.flatten()
    for i in range(5):
        #if region_acro[i] == 'GOMW':
            #data_set = data[i]
            #data_set = data_set[data_set['Year'] != 2023]
        #else:
        data_set = data[i]
        fall_x, fall_y, spring_x, spring_y = scatter_plot_data(dataset=data_set,interest_variable=interest_variable[j])
        axes_flat_spring[i].clear()
        spring_x = np.array(spring_x)
        spring_y = np.array(spring_y)
        mask = ~np.isnan(spring_x) & ~np.isnan(spring_y)
        clean_spring_x = spring_x[mask]
        clean_spring_y = spring_y[mask]

        num_points_spring = len(clean_spring_x)
        has_enough_points_spring = num_points_spring >= 20
        sns.regplot(
            x=clean_spring_x,
            y=clean_spring_y,
            ax=axes_flat_spring[i],
            color=color_options[i],
            fit_reg=has_enough_points_spring
        )
        if 'DOY' in interest_variable[j]:
            axes_flat_spring[i].axhline(y=365, color='gray', linestyle='--', linewidth = 1, alpha=0.5)

        if has_enough_points_spring:
            slope,intercept,r_value,p_value,std_err = stats.linregress(clean_spring_x,clean_spring_y)
            r_square = r_value**2
            stats_text = f"p-value = {p_value:.4f}"
        else:
            stats_text = f"N = {num_points_spring}\n(No trendline)"
        if p_value <=0.0559:
            axes_flat_spring[i].text(
                0.05,0.95,
                stats_text,
                transform=axes_flat_spring[i].transAxes,
                fontsize=15,
                color='red',
                verticalalignment='top',
                fontweight='bold'
            )
        else:
            axes_flat_spring[i].text(
                0.05,0.95,
                stats_text,
                transform=axes_flat_spring[i].transAxes,
                fontsize=15,
                color='black',
                verticalalignment='top'
            )
        axes_flat_spring[i].set_title(titles[i],fontsize=18)
        axes_flat_spring[i].set_xlabel('Year',fontsize=14)
        axes_flat_spring[i].set_ylabel(axis_labels[j],fontsize=13)
        axes_flat_spring[i].set_xticks(x_ticks)
        axes_flat_spring[i].set_xticklabels(x_sparse_labels,rotation=70)
        axes_flat_spring[i].tick_params(labelleft=True)
    axes_flat_spring[5].remove()

    fig_spring.suptitle("Spring Bloom " + variable_title[j] + " by Region",fontsize=20)
    filename = f"Spring_{interest_variable[j]}_scatter"
    plt.tight_layout()
    plt.savefig(rf'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\Figures\Metric_Scatter_Plots\{filename}.png',dpi=300,bbox_inches='tight')
    plt.close(fig_spring)